In [1]:
%cd /home/ubuntu/Project/libero_development
from src.launch_cluster import launch_cluster, shutdown_cluster
cluster, client = launch_cluster(8)

/home/ubuntu/pyvenv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/ubuntu/Project/libero_development
Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-08-24 11:16:22,235 - distributed.deploy.ssh - INFO - 2026-08-24 11:16:22,234 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-08-24 11:16:22,262 - distributed.deploy.ssh - INFO - 2026-08-24 11:16:22,261 - distributed.scheduler - INFO - State start
2026-08-24 11:16:22,263 - distributed.deploy.ssh - INFO - 2026-08-24 11:16:22,263 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-scratch-space/scheduler-m68hngpz', purging
2026-08-24 11:16:22,266 - distributed.deploy.ssh - INFO - 2026-08-24 11:16:22,266 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-08-24 11:16:23,831 - distributed.deploy.ssh - INFO - 2026-08-24 11:16:23,832 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.121:36319'
2026-08-24 11:16:23,832 - distributed.deploy.ssh - INFO - 2026-08-24 11:16:23,834 - distributed.nanny - INFO -

Cluster avviato e connessione stabilita con successo!



In [3]:
from src.data_loader import load_dataset

DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
RAW = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"   # già in cache, niente download
PQ  = "/tmp/kddcup_data.parquet"
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]
X_bag, (mean_ar, std_ar) = load_dataset(
    DATASET_URL_10PC, RAW, PQ, PQ, COL_NAMES,
    n_partitions=64, client=client,
)

Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021


In [4]:
import numpy as np, dask
from src.benchmark import run_benchmark

delayed_parts = [dask.delayed(np.vstack)(p) for p in X_bag.to_delayed()]
X_arr = np.vstack(client.gather(client.compute(delayed_parts)))
print("X:", X_arr.shape)

R = 10
combos = [                      # identiche alla storica
    (8, 32, 1, R),   # under-partitioned
    (8, 64, 1, R),   # balanced (1 part/thread)
    (8, 65, 1, R),   # imbalanced
    (8, 128, 1, R),  # over-partitioned
]
df = run_benchmark(client, X_arr=X_arr, combinations=combos,
                   k_values=[1000], label="b1_validation",
                   max_iter_fit=10, seed=42, averaging_iterations=10)
df

X: (494021, 36)
Testing: k=1000, workers=8, partitions=32, l=1000 (l/k=1), r=10 
 Iterating 10 times.
 -> Cost: 29717.16 | Time: 49.88s
 -> Cost: 29385.19 | Time: 50.89s
 -> Cost: 29635.91 | Time: 52.84s
 -> Cost: 29413.65 | Time: 50.54s
 -> Cost: 29591.44 | Time: 50.56s
 -> Cost: 29440.21 | Time: 50.57s
 -> Cost: 29665.67 | Time: 50.59s
 -> Cost: 29306.48 | Time: 51.06s
 -> Cost: 29434.57 | Time: 49.12s
 -> Cost: 29349.37 | Time: 49.87s
Testing: k=1000, workers=8, partitions=64, l=1000 (l/k=1), r=10 
 Iterating 10 times.
 -> Cost: 29435.49 | Time: 41.51s
 -> Cost: 29455.48 | Time: 42.26s
 -> Cost: 29313.52 | Time: 40.92s
 -> Cost: 29581.27 | Time: 43.10s
 -> Cost: 29588.54 | Time: 43.00s
 -> Cost: 29583.18 | Time: 42.88s
 -> Cost: 29452.83 | Time: 42.68s
 -> Cost: 29551.80 | Time: 41.93s
 -> Cost: 29587.69 | Time: 42.03s
 -> Cost: 29369.81 | Time: 41.36s
Testing: k=1000, workers=8, partitions=65, l=1000 (l/k=1), r=10 
 Iterating 10 times.
 -> Cost: 29573.30 | Time: 42.47s
 -> Cost: 29

,k,l,r,r_effective,partitions,cost,time,seed,workers,l_over_k
0,1000,1000,10,10,32,29717.163510,49.883853,42,8,1
1,1000,1000,10,10,32,29385.188568,50.889342,43,8,1
2,1000,1000,10,10,32,29635.908983,52.838612,44,8,1
3,1000,1000,10,10,32,29413.646967,50.543182,45,8,1
4,1000,1000,10,10,32,29591.437320,50.563115,46,8,1
5,1000,1000,10,10,32,29440.214627,50.566855,47,8,1
6,1000,1000,10,10,32,29665.665970,50.585558,48,8,1
7,1000,1000,10,10,32,29306.476359,51.058117,49,8,1
8,1000,1000,10,10,32,29434.571929,49.123739,50,8,1
9,1000,1000,10,10,32,29349.370250,49.872248,51,8,1


In [5]:
shutdown_cluster(cluster, client)

Cluster e client chiusi.
